In [0]:
%sql

CREATE OR REPLACE TEMPORARY VIEW tvw_dim_productos                              AS
SELECT
     cod_prod                                                                   AS CODIGO_PRODUCTO
    ,desc_prod                                                                  AS DESCRIPCION_PRODUCTO
    ,tip_prod                                                                   AS TIPO_PRODUCTO
    ,tasa_ea                                                                    AS TASA_EA
    ,ROUND(POWER(1 + tasa_ea / 100, 1/12.0) - 1, 6) * 100                       AS TASA_MENSUAL_EQUIVALENTE
    ,plazo_max_meses                                                            AS PLAZO_MAXIMO_MESES
    ,cuota_min                                                                  AS CUOTA_MINIMA
    ,comision_admin                                                             AS COMISION_ADMINISTRATIVA
    ,estado_prod                                                                AS ESTADO_PRODUCTO
    ,CASE
        WHEN LOWER(tip_prod) IN (
            'libre inversion',
            'credito rotativo',
            'tarjeta digital'
        ) THEN 'CREDITO'
        WHEN LOWER(tip_prod) IN (
            'cuenta de ahorro',
            'ahorro digital'
        ) THEN 'AHORRO'
        WHEN LOWER(tip_prod) IN (
            'pago pse',
            'transferencia ach',
            'corresponsalia'
        ) THEN 'TRANSACCIONAL'
        ELSE 'SIN_CLASIFICAR'
     END                                                                        AS FAMILIA_PRODUCTO
    ,CURRENT_DATE()                                                             AS _FECHA_CARGA
    ,'silver.cleaned.tb_productos_cat'                                          AS _FUENTE
FROM silver.cleaned.tb_productos_cat;

In [0]:
%sql
CREATE TABLE IF NOT EXISTS gold.financiero.dim_productos (

    CODIGO_PRODUCTO                     STRING
    ,DESCRIPCION_PRODUCTO               STRING
    ,TIPO_PRODUCTO                      STRING
    ,TASA_EA                            DOUBLE
    ,TASA_MENSUAL_EQUIVALENTE           DOUBLE
    ,PLAZO_MAXIMO_MESES                 LONG
    ,CUOTA_MINIMA                       DOUBLE
    ,COMISION_ADMINISTRATIVA            DOUBLE
    ,ESTADO_PRODUCTO                    STRING
    ,FAMILIA_PRODUCTO                   STRING
    ,_FECHA_CARGA                       DATE
    ,_FUENTE                            STRING

)
USING DELTA
LOCATION 'abfss://gold@stdataknowdeveastus001.dfs.core.windows.net/financiero/dim_productos';

In [0]:
%sql
DELETE FROM gold.financiero.dim_productos;

INSERT INTO gold.financiero.dim_productos
SELECT * FROM tvw_dim_productos;